## 01 - Data audit: Lending Club accepted loans

Technical data quality check of the raw `accepted` dataset: structure, data types, non-loan rows, duplicates and missing values. No target-related analysis.

### 1. Setup

In [1]:
from pathlib import Path

import polars as pl

DATA_RAW = Path("../data/raw")

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

### 2. Load raw data

In [2]:
print(list(DATA_RAW.iterdir()))

[WindowsPath('../data/raw/.gitkeep'), WindowsPath('../data/raw/accepted_2007_to_2018q4.csv'), WindowsPath('../data/raw/accepted_2007_to_2018Q4.csv.gz'), WindowsPath('../data/raw/rejected_2007_to_2018q4.csv'), WindowsPath('../data/raw/rejected_2007_to_2018Q4.csv.gz')]


In [3]:
%%time
# Scan all rows for schema inference: slower, but prevents wrong data types
# in columns that are empty in the first rows of the file.
data_raw = pl.read_csv(
    DATA_RAW / "accepted_2007_to_2018Q4.csv.gz",
    infer_schema_length=None,
)

CPU times: total: 1min 3s
Wall time: 1min 3s


In [4]:
data_raw.glimpse()

Rows: 2260701
Columns: 151
$ id                                         <str> '68407277', '68355089', '68341763', '66310712', '68476807', '68426831', '68476668', '67275481', '68466926', '68616873'
$ member_id                                  <str> null, null, null, null, null, null, null, null, null, null
$ loan_amnt                                  <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ funded_amnt                                <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ funded_amnt_inv                            <f64> 3600.0, 24700.0, 20000.0, 35000.0, 10400.0, 11950.0, 20000.0, 20000.0, 10000.0, 8000.0
$ term                                       <str> ' 36 months', ' 36 months', ' 60 months', ' 60 months', ' 60 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months'
$ int_rate                                   <f64> 13.99, 11.99, 10.78, 14.85, 22.45, 13.

In [5]:
data_raw.head()

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,f64,f64,f64,f64,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,str,str,str,f64,f64,f64
"""68407277""",null,3600.0,3600.0,3600.0,""" 36 months""",13.99,123.03,"""C""","""C4""","""leadman""","""10+ years""","""MORTGAGE""",55000.0,"""Not Verified""","""Dec-2015""","""Fully Paid""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=68407277""",null,"""debt_consolidation""","""Debt consolidation""","""190xx""","""PA""",5.91,0.0,"""Aug-2003""",675.0,679.0,1.0,30.0,null,7.0,0.0,2765.0,29.7,13.0,"""w""",0.0,0.0,4421.723917,4421.72,3600.0,821.72,0.0,0.0,0.0,"""Jan-2019""",122.67,null,"""Mar-2019""",564.0,560.0,0.0,30.0,1.0,"""Individual""",null,null,null,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,null,null

In [6]:
data_raw.tail()

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
str,str,f64,f64,f64,str,f64,f64,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,str,f64,f64,f64,f64,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,f64,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,str,str,str,f64,f64,f64
"""88985880""",null,40000.0,40000.0,40000.0,""" 60 months""",10.49,859.56,"""B""","""B3""","""Vice President ""","""9 years""","""MORTGAGE""",227000.0,"""Verified""","""Oct-2016""","""Current""","""n""","""https://lendingclub.com/browse/loanDetail.action?loan_id=88985880""",null,"""debt_consolidation""",null,"""907xx""","""CA""",12.75,7.0,"""Feb-1995""",705.0,709.0,1.0,9.0,null,5.0,0.0,8633.0,64.9,37.0,"""f""",23252.59,23252.59,24903.93,24903.93,16747.41,8156.52,0.0,0.0,0.0,"""Mar-2019""",859.56,"""Apr-2019""","""Mar-2019""",724.0,720.0,0.0,10.0,1.0,"""Individual""",null,null,null,0.0,0.0,28398.0,0.0,2.0,0.0,1.0,15.0,19765.0,46.0,0.0,0.0,5141.0,51.0,13300.0,3.0,0.0,2.0,2.0,5680.0,4070.0,66.9,0.0,0.0,154.0,258.0,33.0,15.0,3.0,41.0,9.0,1.0,9.0,6.0,2.0,3.0,2.0,15.0,9.0,3.0,23.0,3.0,5.0,0.0,0.0,7.0,0.0,75.7,50.0,0.0,0.0,55970.0,28398.0,12300.0,42670.0,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,null,null,null,null,null,null,null,n

In [7]:
data_raw.shape

(2260701, 151)

In [8]:
data_raw.estimated_size("gb")

2.4276786698028445

### 3. Non-loan rows removal

In [13]:
# Expression reused below:
# True where id can be parsed as an integer
is_numeric_id = pl.col("id").cast(pl.Int64, strict=False).is_not_null()

non_numeric_id = data_raw.filter(
    ~is_numeric_id
)

non_numeric_id.height

33

In [10]:
non_numeric_id.select("id")

id
str
"""Total amount funded in policy code 1: 6417608175"""
"""Total amount funded in policy code 2: 1944088810"""
"""Total amount funded in policy code 1: 1741781700"""
"""Total amount funded in policy code 2: 564202131"""
"""Total amount funded in policy code 1: 1791201400"""
"""Total amount funded in policy code 2: 651669342"""
"""Total amount funded in policy code 1: 1443412975"""
"""Total amount funded in policy code 2: 511988838"""
"""Total amount funded in policy code 1: 2063142975"""


33 rows have a non-numeric `id`. They are not loans: these are summary rows ("Total amount funded in policy code...") and one section header left over from the original LC export.
We will drop these rows and cast `id` to `Int64`.

In [11]:
loans = data_raw.filter(
    is_numeric_id
).with_columns(
    pl.col("id").cast(pl.Int64)
)

print(f"dropped rows: {data_raw.height - loans.height}")
print(f"remaining loans: {loans.height:,}")

assert data_raw.height - loans.height == non_numeric_id.height

dropped rows: 33
remaining loans: 2,260,668


### 4. Loan population checks

In [12]:
loans["loan_status"].value_counts(sort=True)

loan_status,count
str,u32
"""Fully Paid""",1076751
"""Current""",878317
"""Charged Off""",268559
"""Late (31-120 days)""",21467
"""In Grace Period""",8436
"""Late (16-30 days)""",4349
"""Does not meet the credit policy. Status:Fully Paid""",1988
"""Does not meet the credit policy. Status:Charged Off""",761
"""Default""",40


In [14]:
n_unique_ids = loans["id"].n_unique()

print(f"rows: {loans.height:,}, unique ids: {n_unique_ids:,}")
assert n_unique_ids == loans.height, "Duplicate loan ids found"

rows: 2,260,668, unique ids: 2,260,668


In [18]:
null_summary = (
    loans.null_count()
    .unpivot(variable_name="column", value_name="null_count")
    .with_columns(
        (pl.col("null_count") / loans.height * 100).round(2).alias("null_pct")
    )
    .sort("null_count", descending=True)
)

with pl.Config(tbl_rows=-1):
    print(null_summary)

shape: (151, 3)
┌────────────────────────────────────────────┬────────────┬──────────┐
│ column                                     ┆ null_count ┆ null_pct │
│ ---                                        ┆ ---        ┆ ---      │
│ str                                        ┆ u32        ┆ f64      │
╞════════════════════════════════════════════╪════════════╪══════════╡
│ member_id                                  ┆ 2260668    ┆ 100.0    │
│ orig_projected_additional_accrued_interest ┆ 2252017    ┆ 99.62    │
│ hardship_type                              ┆ 2249751    ┆ 99.52    │
│ hardship_reason                            ┆ 2249751    ┆ 99.52    │
│ hardship_status                            ┆ 2249751    ┆ 99.52    │
│ deferral_term                              ┆ 2249751    ┆ 99.52    │
│ hardship_amount                            ┆ 2249751    ┆ 99.52    │
│ hardship_start_date                        ┆ 2249751    ┆ 99.52    │
│ hardship_end_date                          ┆ 2249751    ┆ 9

In [23]:
unique_summary = (
    loans.select(pl.all().n_unique())
    .unpivot(variable_name="column", value_name="n_unique")
    .sort("n_unique")
)

unique_summary.head(15)

column,n_unique
str,u32
"""member_id""",1
"""policy_code""",1
"""term""",2
"""pymnt_plan""",2
"""initial_list_status""",2
"""application_type""",2
"""hardship_flag""",2
"""hardship_type""",2
"""deferral_term""",2


In [26]:
constant_cols = (
    unique_summary
    .filter(pl.col("n_unique") == 1)
    .get_column("column")
    .to_list()
)

print(constant_cols)
print(loans["policy_code"].value_counts())

['member_id', 'policy_code']
shape: (1, 2)
┌─────────────┬─────────┐
│ policy_code ┆ count   │
│ ---         ┆ ---     │
│ f64         ┆ u32     │
╞═════════════╪═════════╡
│ 1.0         ┆ 2260668 │
└─────────────┴─────────┘


In [27]:
binary_cols = (
    unique_summary
    .filter(pl.col("n_unique") == 2)
    .get_column("column")
    .to_list()
)

for col in binary_cols:
    print(loans[col].value_counts(sort=True))

shape: (2, 2)
┌────────────┬─────────┐
│ term       ┆ count   │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│  36 months ┆ 1609754 │
│  60 months ┆ 650914  │
└────────────┴─────────┘
shape: (2, 2)
┌────────────┬─────────┐
│ pymnt_plan ┆ count   │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ n          ┆ 2260048 │
│ y          ┆ 620     │
└────────────┴─────────┘
shape: (2, 2)
┌─────────────────────┬─────────┐
│ initial_list_status ┆ count   │
│ ---                 ┆ ---     │
│ str                 ┆ u32     │
╞═════════════════════╪═════════╡
│ w                   ┆ 1535467 │
│ f                   ┆ 725201  │
└─────────────────────┴─────────┘
shape: (2, 2)
┌──────────────────┬─────────┐
│ application_type ┆ count   │
│ ---              ┆ ---     │
│ str              ┆ u32     │
╞══════════════════╪═════════╡
│ Individual       ┆ 2139958 │
│ Joint App        ┆ 120710  │
└──────────────────┴─────────┘
shape: (2, 2)
┌─────────

In [28]:
# NOTE: member_id is all null and policy_code is constant (1.0) -> drop both.
# NOTE: no "policy code 2" loans in this dataset (policy_code = 1 for all).
# NOTE: term has a leading space (" 36 months") -> handle in type conversion.
# NOTE: hardship_type, deferral_term, hardship_length: single value + nulls
#       (10,917 loans with hardship; post-origination info).
# NOTE: hardship_flag = Y for 832 loans only vs. 10,917 with hardship_* filled
#       -> check meaning (active at snapshot vs. ever?).
# NOTE: debt_settlement_flag = Y (34,246) matches non-null settlement_* -> consistent.
# NOTE: Joint App (120,710) matches non-null annual_inc_joint -> consistent;
#       sec_app_* non-null for 108,021 only -> check over time.
# NOTE: pymnt_plan = y for 620 loans only (near-constant, decide on train).

In [29]:
content_duplicates = loans.drop("id", "url").is_duplicated()

print(f"content duplicates: {content_duplicates.sum():,}")

content duplicates: 0


In [35]:
DATE_COLS = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "sec_app_earliest_cr_line",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "debt_settlement_flag_date",
    "settlement_date",
]

In [37]:
dates_parsed = loans.select(
    pl.col(DATE_COLS).str.to_date("%b-%Y", strict=False)
)

date_check = pl.DataFrame({
    "column": DATE_COLS,
    "nulls_before": [loans[col].null_count() for col in DATE_COLS],
    "nulls_after": [dates_parsed[col].null_count() for col in DATE_COLS],
}).with_columns(
    (pl.col("nulls_after") - pl.col("nulls_before")).alias("parse_failures")
)

date_check

column,nulls_before,nulls_after,parse_failures
str,i64,i64,i64
"""issue_d""",0,0,0
"""earliest_cr_line""",29,29,0
"""last_pymnt_d""",2427,2427,0
"""next_pymnt_d""",1345310,1345310,0
"""last_credit_pull_d""",72,72,0
"""sec_app_earliest_cr_line""",2152647,2152647,0
"""hardship_start_date""",2249751,2249751,0
"""hardship_end_date""",2249751,2249751,0
"""payment_plan_start_date""",2249751,2249751,0
